[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C36_GPU_Kernels_Course/05_flash_attention/05_flash_attention.ipynb)

# 05 · FlashAttention 内核（用 numpy 从零写出）

这是全课的**集大成**。我们把模块 03 的**分块**与模块 04 的**在线 softmax**拼起来，再加一个**输出累加器**，
就得到 FlashAttention —— 一个**永不 materialize n×n 分数矩阵**、显存 O(n²)→O(n)、却与朴素注意力**逐位相同**的内核。

**路线**：
1. 朴素注意力（参考实现，会算出完整 n×n 的 S）
2. 在线 softmax 复习 + 加输出累加器 → 两块合并
3. **FlashAttention 前向**（非因果）→ 对拍朴素，`atol=1e-10`
4. 显存账：O(n²) vs O(n)
5. **因果掩码版**（含防御性 guard 防 NaN）→ 对拍朴素因果
6. IO/HBM 账
7. ✏️ 练习（自己写 flash 前向 / 合并状态 / 因果）
8. 📖 答案 · 🧪 真实数据胶囊 · 🔧 Triton 伪代码

> **本课纪律**：每个内核都用 `np.allclose(flash, naive)` 对拍参考。结构正确 → 数值一致 → 逻辑可迁移到 Triton/CUDA。

## 1 · 朴素注意力（参考实现）

$O=\mathrm{softmax}(QK^\top/\sqrt{d})\,V$。朴素实现把整个 $n\times n$ 的分数矩阵 $S$ 和概率矩阵 $P$ **materialize 出来**——
这正是长序列爆显存的根源。它作为我们对拍的 **ground truth**。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def softmax_rows(S):
    # 数值稳定 softmax：每行减最大值再 exp（模块 04）
    m = S.max(axis=-1, keepdims=True)
    e = np.exp(S - m)
    return e / e.sum(axis=-1, keepdims=True)

def attention_naive(Q, K, V, causal=False):
    n, d = Q.shape
    S = Q @ K.T / np.sqrt(d)          # (n, n)  <- 完整分数矩阵，O(n^2) 显存
    if causal:
        mask = np.triu(np.ones((n, n), dtype=bool), k=1)   # 上三角=未来
        S = np.where(mask, -np.inf, S)
    P = softmax_rows(S)              # (n, n)  <- 完整概率矩阵
    return P @ V                     # (n, d)

n, d = 6, 4
Q = rng.standard_normal((n, d)); K = rng.standard_normal((n, d)); V = rng.standard_normal((n, d))
O_ref = attention_naive(Q, K, V)
print('朴素注意力输出形状:', O_ref.shape)
# 概率矩阵每行和为 1
S = Q @ K.T / np.sqrt(d)
assert np.allclose(softmax_rows(S).sum(axis=1), 1.0)
print('softmax 每行和为 1 ✅  （但我们刚刚 materialize 了整个', S.shape, '矩阵）')

## 2 · 在线 softmax + 输出累加器：两块合并

模块 04 的在线 softmax 流式维护 `m`（运行最大值）和 `l`（运行指数和）。
FlashAttention 多维护一个 **`O`（运行加权输出累加器）**。**唯一的新东西**：每次最大值刷新，
不仅 `l` 要乘校正因子 `corr=exp(m_old-m_new)` 回缩，**`O` 也要乘同一个 `corr`**——因为 `O` 里已累加的项是以旧 `m` 为基准的。

先把「合并一个新块」写成函数，再验证：把 K/V 切两半分别合并 == 朴素。

In [ ]:
def merge_block(O, l, m, Sblock, Vblock):
    '''把一个新 KV 块的贡献合并进运行状态 (O, l, m)。
       Sblock: (n, bk) 已缩放的分数; Vblock: (bk, d)。返回更新后的 (O, l, m)。'''
    m_block = Sblock.max(axis=1, keepdims=True)          # (n,1) 本块局部最大
    m_new   = np.maximum(m, m_block)                     # (n,1) 新的运行最大
    safe    = np.where(np.isinf(m_new), 0.0, m_new)      # 防 -inf 减 -inf
    P       = np.exp(Sblock - safe)                      # (n, bk) 对齐到新基准
    P       = np.where(np.isinf(Sblock), 0.0, P)         # 被掩元素概率=0
    corr    = np.where(np.isinf(m), 0.0, np.exp(m - safe))   # (n,1) 校正因子
    l = corr * l + P.sum(axis=1, keepdims=True)          # 缩放旧 l + 本块和
    O = corr * O + P @ Vblock                            # 缩放旧 O + 本块加权
    return O, l, m_new

n, d = 12, 4
Q = rng.standard_normal((n, d)); K = rng.standard_normal((n, d)); V = rng.standard_normal((n, d))
scale = 1.0 / np.sqrt(d)
O_ref = attention_naive(Q, K, V)

# 流式合并两半 KV
O = np.zeros((n, d)); l = np.zeros((n, 1)); m = np.full((n, 1), -np.inf)
for j0 in [0, 6]:
    Sb = (Q @ K[j0:j0+6].T) * scale          # (n, 6) 只算这一小块
    O, l, m = merge_block(O, l, m, Sb, V[j0:j0+6])
out = O / l                                   # 最后一次性归一化
print('两块合并 vs 朴素 最大误差:', np.abs(out - O_ref).max())
assert np.allclose(out, O_ref, atol=1e-10)
print('✅ 在线 softmax + 输出累加器：分两块流式合并 == 朴素一次算完')

## 3 · FlashAttention 前向（非因果）

把第 2 节的合并放进一个循环：**query 全程在外（FA-2 风格），K/V 块在内层流式扫过**。
运行状态 `(O, l, m)` 一直留着，**完整的 n×n 分数矩阵从未存在过**——每次只算 `(n, 块宽)` 的一小条。

In [ ]:
def flash_attention(Q, K, V, block_kv=16, causal=False):
    n, d = Q.shape
    scale = 1.0 / np.sqrt(d)
    O = np.zeros((n, d))               # 运行输出累加器
    m = np.full((n, 1), -np.inf)       # 运行最大值
    l = np.zeros((n, 1))               # 运行归一化和
    for j0 in range(0, n, block_kv):                 # 流式扫 K/V 块
        Kj = K[j0:j0+block_kv]                        # (bk, d) 载入 SRAM
        Vj = V[j0:j0+block_kv]
        Sij = (Q @ Kj.T) * scale                     # (n, bk) 小分数块，绝非 n×n
        if causal:                                   # 因果掩码：key 在 query 之后则屏蔽
            qi = np.arange(n)[:, None]
            kj = (j0 + np.arange(Sij.shape[1]))[None, :]
            Sij = np.where(kj > qi, -np.inf, Sij)
        # ---- 在线 softmax 更新（含防御性 guard）----
        m_block = Sij.max(axis=1, keepdims=True)
        m_new   = np.maximum(m, m_block)
        safe    = np.where(np.isinf(m_new), 0.0, m_new)
        P       = np.exp(Sij - safe)
        P       = np.where(np.isinf(Sij), 0.0, P)
        corr    = np.where(np.isinf(m), 0.0, np.exp(m - safe))
        l = corr * l + P.sum(axis=1, keepdims=True)
        O = corr * O + P @ Vj
        m = m_new
    return O / np.where(l == 0, 1.0, l)

n, d = 20, 8
Q = rng.standard_normal((n, d)); K = rng.standard_normal((n, d)); V = rng.standard_normal((n, d))
O_ref = attention_naive(Q, K, V)
for bk in [1, 7, 8, 16, n]:                 # 各种块宽，含不整除 n 的 7
    out = flash_attention(Q, K, V, block_kv=bk)
    err = np.abs(out - O_ref).max()
    assert np.allclose(out, O_ref, atol=1e-10), (bk, err)
    print(f'block_kv={bk:>2d}: 与朴素最大误差 {err:.2e}  ✅')
print('\n关键：不论块宽怎么切，结果都与朴素逐位相同（浮点误差内）—— 分块没有引入近似。')

**为什么块宽无关结果？** 因为校正因子的「望远镜相消」（见讲解第 5 节）：每一项最终都被精确对齐到
全局最大值这个统一基准。分块只改变*计算顺序*，不改变*数学结果*。这就是 FlashAttention 是**精确**注意力的含义。

## 4 · 显存账：O(n²) vs O(n)

朴素被 $n\times n$ 的 S 支配 → $\Theta(n^2)$；FlashAttention 只需输出 `O`、运行量 `m,l`、当前一个块 → 在 $n^2$ 量级上是 $\Theta(n)$。

In [ ]:
def attn_peak_mem_naive(n, d, b=4):
    return b * (n * n + n * d)            # S 矩阵(n*n) 支配

def attn_peak_mem_flash(n, d, block_kv, b=4):
    return b * (n * d + n + block_kv * d) # O(n*d) + m,l(n) + 当前块

print(f"{'n':>8s} {'朴素(MB)':>12s} {'Flash(MB)':>12s} {'倍数':>8s}")
for n in [512, 2048, 8192, 32768]:
    mn = attn_peak_mem_naive(n, 128)
    mf = attn_peak_mem_flash(n, 128, block_kv=128)
    print(f'{n:>8d} {mn/1e6:>12.1f} {mf/1e6:>12.3f} {mn/mf:>7.0f}x')
assert attn_peak_mem_flash(32768, 128, 128) < attn_peak_mem_naive(32768, 128)
print('\n✅ n 越大差距越夸张：朴素随 n 平方膨胀，Flash 随 n 线性 —— 这就是长上下文的钥匙。')

## 5 · 因果掩码版（含防 NaN 的 guard）

自回归模型里每个 query 只能看自己及之前的 key。我们已在 `flash_attention(..., causal=True)` 里实现：
对每个 KV 块，把 `key_index > query_index` 的位置置 `-inf`。

**陷阱**：若某行在某块里所有 key 都被掩成 `-inf`、且此前也没见过可见 key，则 `exp(-inf-(-inf))=exp(nan)`=NaN。
我们的 guard（`safe` 把 `-inf` 的 `m_new` 换 0、被掩元素概率置 0、`corr` 对 `-inf` 的 `m` 置 0）正是为此。

In [ ]:
for (n, d) in [(4, 2), (16, 8), (33, 8)]:        # 含不整除的 33
    Q = rng.standard_normal((n, d)); K = rng.standard_normal((n, d)); V = rng.standard_normal((n, d))
    O_ref = attention_naive(Q, K, V, causal=True)
    for bk in [1, 7, 16, n]:
        out = flash_attention(Q, K, V, block_kv=bk, causal=True)
        assert not np.isnan(out).any(), f'NaN! n={n} bk={bk}'
        assert np.allclose(out, O_ref, atol=1e-10), f'mismatch n={n} bk={bk}'
    print(f'n={n:>2d} d={d}: 因果版所有块宽都 == 朴素因果，且无 NaN ✅')
print('\n✅ 因果 FlashAttention 正确。对角块逐元素掩码、纯未来块整块被屏蔽（真实内核会直接跳过省一半算力）。')

## 6 · IO/HBM 账：为什么更快

FlashAttention 的 FLOPs 并不比朴素少（甚至略多），它快是因为 **HBM 访问少一个数量级**。
朴素要把 $n\times n$ 的 S、P 写出再读回（$\Theta(n^2)$）；Flash 只读 Q,K,V、写 O（$\Theta(nd)$ 级，分块复用后总量 $\Theta(n^2d^2/M)$）。

In [ ]:
def hbm_bytes_naive(n, d, b=4):
    # 读 Q,K,V(3nd) + 写 S(n^2) + 读 S 写 P(2n^2) + 读 P(n^2) + 写 O(nd)
    return b * (3*n*d + n*n + 2*n*n + n*n + n*d)

def hbm_bytes_flash(n, d, b=4):
    # 读 Q,K,V 各一次(分块复用) + 写 O；n^2 量级的 S/P 从不落 HBM
    return b * (3*n*d + n*d)

print(f"{'n':>8s} {'朴素HBM(MB)':>14s} {'FlashHBM(MB)':>14s} {'倍数':>8s}")
for n in [512, 2048, 8192, 32768]:
    hn, hf = hbm_bytes_naive(n, 128), hbm_bytes_flash(n, 128)
    print(f'{n:>8d} {hn/1e6:>14.1f} {hf/1e6:>14.3f} {hn/hf:>7.0f}x')
assert hbm_bytes_flash(8192, 128) < hbm_bytes_naive(8192, 128)
print('\n✅ FLOPs 几乎一样，HBM 往返却差一个数量级 —— 访存受限算子上，省 IO 远比省 FLOPs 值钱（模块 00 的世界观）。')

---
## ✏️ 练习 1：从零写 FlashAttention 前向（非因果）

给你运行状态和循环骨架，**自己填写在线 softmax 的递推**（第 3、4、5 步）。
目标：对各种块宽都与 `attention_naive` 逐位相同。这是本模块的核心判分点。

递推（讲解第 4 节）：`m_new=max(m,max(Sij))`；`P=exp(Sij-m_new)`；`corr=exp(m-m_new)`；
`l=corr*l+sum(P)`；`O=corr*O+P@Vj`；最后 `O/l`。

In [ ]:
def my_flash(Q, K, V, block_kv=16):
    n, d = Q.shape
    scale = 1.0 / np.sqrt(d)
    O = np.zeros((n, d))
    m = np.full((n, 1), -np.inf)
    l = np.zeros((n, 1))
    for j0 in range(0, n, block_kv):
        Kj = K[j0:j0+block_kv]; Vj = V[j0:j0+block_kv]
        Sij = (Q @ Kj.T) * scale             # (n, bk)
        # TODO: 用在线 softmax 更新 m, l, O（非因果，不必加 guard）
        #   1) m_new = 逐行 max(m, Sij 的行最大)
        #   2) P = exp(Sij - m_new)
        #   3) corr = exp(m - m_new)
        #   4) l = corr*l + P 的行和 ;  O = corr*O + P @ Vj
        #   5) m = m_new
        raise NotImplementedError
    return O / l

In [ ]:
# —— 练习 1 自测 ——
for (n, d) in [(10, 4), (20, 8)]:
    Q = rng.standard_normal((n, d)); K = rng.standard_normal((n, d)); V = rng.standard_normal((n, d))
    ref = attention_naive(Q, K, V)
    for bk in [1, 4, 8, n]:
        out = my_flash(Q, K, V, block_kv=bk)
        assert np.allclose(out, ref, atol=1e-10), f'n={n} bk={bk} 误差 {np.abs(out-ref).max():.2e}'
print('✅ 练习 1 通过：你的 FlashAttention 前向与朴素逐位相同（任意块宽）')

## ✏️ 练习 2：合并两段注意力状态（FlashDecoding 的核心）

推理解码时常把 KV 序列**切成几段并行**算各自的 `(O, l, m)`，再**合并**。实现 `combine_two(O1,l1,m1, O2,l2,m2)`：
把两段（已各自归一化*之前*的）运行状态合并成一段。

合并规则：`m=max(m1,m2)`；`O = exp(m1-m)*O1 + exp(m2-m)*O2`；`l = exp(m1-m)*l1 + exp(m2-m)*l2`。

In [ ]:
def flash_partial(Q, K, V, j_start, j_end):
    '''只在 K/V 的 [j_start:j_end) 段上算运行状态（归一化前），返回 (O, l, m)。'''
    n, d = Q.shape; scale = 1.0/np.sqrt(d)
    Sij = (Q @ K[j_start:j_end].T) * scale
    m = Sij.max(axis=1, keepdims=True)
    P = np.exp(Sij - m)
    l = P.sum(axis=1, keepdims=True)
    O = P @ V[j_start:j_end]
    return O, l, m

def combine_two(O1, l1, m1, O2, l2, m2):
    # TODO: 合并两段运行状态（见上式），返回 (O, l, m)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
n, d = 16, 8
Q = rng.standard_normal((n, d)); K = rng.standard_normal((n, d)); V = rng.standard_normal((n, d))
ref = attention_naive(Q, K, V)
O1, l1, m1 = flash_partial(Q, K, V, 0, 9)        # 第一段
O2, l2, m2 = flash_partial(Q, K, V, 9, n)        # 第二段
O, l, m = combine_two(O1, l1, m1, O2, l2, m2)
out = O / l
assert np.allclose(out, ref, atol=1e-10), np.abs(out-ref).max()
print('✅ 练习 2 通过：两段并行算 + 合并 == 朴素（这正是 FlashDecoding 的并行方式）')

## ✏️ 练习 3：给 FlashAttention 加因果掩码

在练习 1 的非因果版上加因果掩码 + 防 NaN guard。对每个 KV 块，把 `key_index > query_index` 置 `-inf`，
并加 guard（`safe` 替换 `-inf` 的 `m_new`、被掩元素 `P` 置 0、`corr` 对 `-inf` 的 `m` 置 0）。
目标：与 `attention_naive(causal=True)` 一致且无 NaN。

In [ ]:
def my_flash_causal(Q, K, V, block_kv=16):
    n, d = Q.shape; scale = 1.0/np.sqrt(d)
    O = np.zeros((n, d)); m = np.full((n,1), -np.inf); l = np.zeros((n,1))
    for j0 in range(0, n, block_kv):
        Kj = K[j0:j0+block_kv]; Vj = V[j0:j0+block_kv]
        Sij = (Q @ Kj.T) * scale
        # TODO:
        #   (a) 因果掩码：qi=arange(n)[:,None], kj=(j0+arange(bk))[None,:]; kj>qi 处置 -inf
        #   (b) m_new=max(m, Sij行最大); safe=where(isinf(m_new),0,m_new)
        #   (c) P=exp(Sij-safe); P=where(isinf(Sij),0,P)
        #   (d) corr=where(isinf(m),0,exp(m-safe))
        #   (e) l=corr*l+P行和; O=corr*O+P@Vj; m=m_new
        raise NotImplementedError
    return O / np.where(l==0, 1.0, l)

In [ ]:
# —— 练习 3 自测 ——
for (n, d) in [(8, 4), (33, 8)]:
    Q = rng.standard_normal((n, d)); K = rng.standard_normal((n, d)); V = rng.standard_normal((n, d))
    ref = attention_naive(Q, K, V, causal=True)
    for bk in [1, 7, n]:
        out = my_flash_causal(Q, K, V, block_kv=bk)
        assert not np.isnan(out).any(), f'NaN n={n} bk={bk}'
        assert np.allclose(out, ref, atol=1e-10), f'n={n} bk={bk}'
print('✅ 练习 3 通过：因果 FlashAttention 正确且无 NaN')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def my_flash(Q, K, V, block_kv=16):
    n, d = Q.shape; scale = 1.0/np.sqrt(d)
    O = np.zeros((n, d)); m = np.full((n,1), -np.inf); l = np.zeros((n,1))
    for j0 in range(0, n, block_kv):
        Kj = K[j0:j0+block_kv]; Vj = V[j0:j0+block_kv]
        Sij = (Q @ Kj.T) * scale
        m_new = np.maximum(m, Sij.max(axis=1, keepdims=True))
        P = np.exp(Sij - m_new)
        corr = np.exp(m - m_new)
        l = corr * l + P.sum(axis=1, keepdims=True)
        O = corr * O + P @ Vj
        m = m_new
    return O / l

In [ ]:
# 练习 2 参考答案
def combine_two(O1, l1, m1, O2, l2, m2):
    m = np.maximum(m1, m2)
    a1 = np.exp(m1 - m); a2 = np.exp(m2 - m)
    O = a1 * O1 + a2 * O2
    l = a1 * l1 + a2 * l2
    return O, l, m

In [ ]:
# 练习 3 参考答案
def my_flash_causal(Q, K, V, block_kv=16):
    n, d = Q.shape; scale = 1.0/np.sqrt(d)
    O = np.zeros((n, d)); m = np.full((n,1), -np.inf); l = np.zeros((n,1))
    for j0 in range(0, n, block_kv):
        Kj = K[j0:j0+block_kv]; Vj = V[j0:j0+block_kv]
        Sij = (Q @ Kj.T) * scale
        qi = np.arange(n)[:, None]
        kj = (j0 + np.arange(Sij.shape[1]))[None, :]
        Sij = np.where(kj > qi, -np.inf, Sij)
        m_new = np.maximum(m, Sij.max(axis=1, keepdims=True))
        safe = np.where(np.isinf(m_new), 0.0, m_new)
        P = np.exp(Sij - safe); P = np.where(np.isinf(Sij), 0.0, P)
        corr = np.where(np.isinf(m), 0.0, np.exp(m - safe))
        l = corr * l + P.sum(axis=1, keepdims=True)
        O = corr * O + P @ Vj
        m = m_new
    return O / np.where(l == 0, 1.0, l)

---
## 🧪 真实数据胶囊：为什么 FlashAttention 解锁了长上下文

用真实配置算一笔账：**单个注意力头**的 $n\times n$ 分数矩阵（FP16，2 字节）有多大？再乘上一层的头数（Llama-7B 是 32 头）。
对比 H100 的 80GB 显存，看朴素注意力在什么长度就**装不下**了——而 FlashAttention 显存只随 $n$ 线性、根本不需要存这些矩阵。

In [ ]:
def score_matrix_gb(n, b=2):
    # TODO: 返回 n×n 分数矩阵的大小（GB）。b=每元素字节(FP16=2)。1 GB = 1e9 字节
    raise NotImplementedError

In [ ]:
# 自测
H100_MEM_GB = 80
N_HEADS = 32          # Llama-7B 每层 32 个注意力头（朴素实现每个头各一个 n×n 矩阵）
print(f"{'序列长 n':>10s} {'单头 S(GB)':>12s} {'32 头合计(GB)':>16s} {'装得进 H100?':>16s}")
for n in [2048, 8192, 32768, 131072]:
    gb1 = score_matrix_gb(n)
    gbH = gb1 * N_HEADS
    fits = 'yes' if gbH < H100_MEM_GB else 'NO (爆显存)'
    print(f'{n:>10d} {gb1:>12.3f} {gbH:>16.1f} {fits:>16s}')
assert abs(score_matrix_gb(32768) - 2.147) < 0.01            # 32768^2 * 2 / 1e9
assert score_matrix_gb(32768) * N_HEADS > 0.8 * H100_MEM_GB  # 32k×32头 ~69GB，逼近 80GB
assert score_matrix_gb(131072) * N_HEADS > H100_MEM_GB       # 128k×32头 远超 80GB
print('\n✅ 胶囊通过：32k 上下文时，仅一层的 32 个分数矩阵合计就 ~69GB，逼近 H100 上限；')
print('   128k 时一层就要 ~1100GB（还没算批量与反向）—— 朴素注意力的 n² 显存是长上下文的死穴。')
print('   FlashAttention 永不存这些矩阵（显存 O(n)）→ 长上下文从此可行（接 C25）。')

In [ ]:
# 📖 胶囊参考答案
def score_matrix_gb(n, b=2):
    return b * n * n / 1e9

---
## 🔧 旁注：对应的 Triton FlashAttention 内核长什么样

你刚写的 numpy（query 在外、K/V 块在内、运行 `m/l/acc`、最后归一化）就是真实 Triton FlashAttention 前向的**结构**（伪代码，**本环境不跑**）：

```python
import triton, triton.language as tl

@triton.jit
def flash_fwd(Q, K, V, O, n, d, scale, BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    pid_m = tl.program_id(0)                       # 一个 program 负责一个 query 块
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    q = tl.load(Q + offs_m[:, None]*d + tl.arange(0, d)[None, :])   # 载入 query 块到 SRAM
    m_i = tl.full([BLOCK_M], -float('inf'), tl.float32)   # 运行最大值
    l_i = tl.zeros([BLOCK_M], tl.float32)                 # 运行归一化和
    acc = tl.zeros([BLOCK_M, d], tl.float32)              # 运行输出累加器
    for j0 in range(0, n, BLOCK_N):                       # 流式扫 K/V 块
        k = tl.load(...); v = tl.load(...)                # 载入 K/V 块到 SRAM
        s = tl.dot(q, tl.trans(k)) * scale                # (BLOCK_M, BLOCK_N) 小分数块
        m_new = tl.maximum(m_i, tl.max(s, axis=1))        # 在线 softmax 更新
        p = tl.exp(s - m_new[:, None])
        alpha = tl.exp(m_i - m_new)                       # 校正因子
        l_i = alpha * l_i + tl.sum(p, axis=1)
        acc = acc * alpha[:, None] + tl.dot(p, v)         # 缩放 + 累加（== 我们的 corr*O + P@Vj）
        m_i = m_new
    acc = acc / l_i[:, None]                              # 最后归一化
    tl.store(O + offs_m[:, None]*d + tl.arange(0, d)[None, :], acc)

# 启动: grid = (triton.cdiv(n, BLOCK_M),)
# 真实版还会用 @triton.autotune 搜 BLOCK_M/BLOCK_N、加因果跳块、Tensor Core(tl.dot)
```

逐行对应：`m_i/l_i/acc` ↔ 我们的 `m/l/O`；`alpha` ↔ `corr`；`acc*alpha[:,None]+tl.dot(p,v)` ↔ `corr*O + P@Vj`。
**这就是 FA-2 的结构**——query 在外层、KV 块在内层。你在 numpy 里验证过的递推，可以几乎一对一搬过去。

### 小结
- FlashAttention = **分块（03）+ 在线 softmax（04）+ 输出累加器**；唯一新东西是用校正因子 `corr` 同时缩放 `l` 和 `O`。
- **精确**（非近似）：校正因子望远镜式相消，每项被对齐到全局最大值 → 与朴素逐位相同（`atol=1e-10` 对拍通过）。
- **显存 O(n²)→O(n)**：永不 materialize n×n 的 S；**HBM 访问** O(n²)→O(n²d²/M)，这才是它快的真正原因（FLOPs 反而略多）。
- 因果掩码：对角块逐元素掩、纯未来块整块跳过；注意防 NaN 的 guard。
- FA-2 把 query 放外层（本课实现的结构）、FA-3 用异步+FP8；FlashDecoding 沿 KV 切分并合并状态（练习 2）。

🎉 **恭喜你读完并写完了整门课**。你现在不止会调用 FlashAttention——你**会写它**，并理解它为什么快、为什么对。
下一步：把这些验证过的逻辑写成真实 `@triton.jit` 内核（references 里的 Triton 教程），接 C24 推理服务 / C25 长上下文。